# Deployable training & artifact export

This notebook trains the Random Forest, computes per-row risk and explanations, and writes deployable artifacts for the backend.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print('Imports OK')

In [ ]:
# Paths and constants
DATA = Path('..') / 'dataset' / 'cleaned' / 'Instagram_cleaned.csv'
ARTIFACTS = Path('..') / 'artifacts'
DEPLOY_DIR = ARTIFACTS / 'deployment'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
RISK_THRESHOLDS = { 'low': 30, 'medium': 70, 'high': 100 }
print('Paths set:', DATA, DEPLOY_DIR)

In [ ]:
# Load data
df = pd.read_csv(DATA)
print('Loaded', len(df), 'rows')
assert 'fake' in df.columns, 'Target `fake` missing'

# Feature selection (same logic as scripts)
feature_cols = [
    c for c in df.columns
    if (c.endswith('_capped') or c.startswith('log_') or c.endswith('_clipped')) and c not in ['fake', 'fake_capped']
]
for flag in ['profile_pic','private','external_url','name_eq_username']:
    if flag in df.columns and flag not in feature_cols:
        feature_cols.append(flag)
print('Using', len(feature_cols), 'features')

In [ ]:
# Rule-based explanation function (deployable)
def generate_reasons(row):
    reasons = []
    if row.get('profile_pic') == 0:
        reasons.append('No profile picture')
    if row.get('username_num_length_capped', 0) > 0.5:
        reasons.append('Digit-heavy username')
    if row.get('description_length_capped') == 0:
        reasons.append('Empty bio')
    if row.get('followers_following_ratio_clipped', 1.0) < 0.2:
        reasons.append('Suspicious follower/following ratio')
    if row.get('num_posts_capped', 9999) < 5:
        reasons.append('Very low posting activity')
    return reasons
print('Explanation function ready')

In [ ]:
# Train model using selected features
X = df[feature_cols].fillna(0)
y = df['fake'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:,1]
metrics = dict(accuracy=accuracy_score(y_test,y_pred), precision=precision_score(y_test,y_pred,zero_division=0), recall=recall_score(y_test,y_pred,zero_division=0), f1=f1_score(y_test,y_pred,zero_division=0), roc_auc=roc_auc_score(y_test,y_proba))
print('Training metrics:', metrics)

In [ ]:
# Compute per-row risk and explanations and save deployable artifacts
probs = rf.predict_proba(df[feature_cols].fillna(0))[:,1]
df['fraud_risk_prob'] = probs
df['fraud_risk_percent'] = (df['fraud_risk_prob'] * 100).round(2)
outlier_cols = [c for c in df.columns if c.endswith('_outlier_p')]
df['outlier_count'] = df[outlier_cols].sum(axis=1) if outlier_cols else 0
df['suspicion_score'] = np.minimum(100, df['fraud_risk_percent'] + 5 * df['outlier_count'])
df['reasons'] = df.apply(generate_reasons, axis=1)
df['reasons'] = df['reasons'].apply(lambda x: '; '.join(x) if isinstance(x,(list,tuple)) and x else '')
# save augmented CSV
out_csv = Path('..') / 'dataset' / 'cleaned' / 'Instagram_with_risk_scores.csv'
df.to_csv(out_csv, index=False)
print('Saved augmented CSV to', out_csv)
# save deployment artifacts
model_path = DEPLOY_DIR / 'fake_profile_detector.pkl'
features_path = DEPLOY_DIR / 'model_features.pkl'
joblib.dump(rf, model_path)
joblib.dump(feature_cols, features_path)
(DEPLOY_DIR / 'risk_thresholds.json').write_text(json.dumps(RISK_THRESHOLDS, indent=2))
(DEPLOY_DIR / 'explanation.py').write_text('''def generate_reasons(row):
    reasons = []
    if row.get('profile_pic') == 0:
        reasons.append('No profile picture')
    if row.get('username_num_length_capped', 0) > 0.5:
        reasons.append('Digit-heavy username')
    if row.get('description_length_capped') == 0:
        reasons.append('Empty bio')
    if row.get('followers_following_ratio_clipped', 1.0) < 0.2:
        reasons.append('Suspicious follower/following ratio')
    if row.get('num_posts_capped', 9999) < 5:
        reasons.append('Very low posting activity')
    return reasons
''')
print('Saved deployment artifacts to', DEPLOY_DIR)

In [ ]:
# Show sample of the new columns
df[['fraud_risk_percent','suspicion_score','reasons']].head(10)